# Mapping Toronto's Urban Heat at Higher Resolution: Machine Learning–Based Downscaling of ECOSTRESS LST from 70 m to 10 m in Python

by Aycha Tammour

In this notebook, i demostrate the process of downscaling ECOSTRESS Land Surface Temperature (LST) data from its native 70m resolution to 10m. I follow the approach presented in a NASA ARSET training module using Google Earth Engine, but implement it in Python using open-source libraries and data.



## How the downscaling work?

The idea is simple, we know that the low resolution ECOSTRESS LST data is correlated with the higher resolution reflectance data. This correlation means that if we have a sample of the LST data and reflectance data for the same AOI, we can train a machine learning model to learn the relationship between the two. Once we have a trained model, we can use it to predict the LST values at the higher resolution using the reflectance data as input.

## Location and Dates

### Area of Interest: Toronto, Canada

We will focus on Toronto...

In the summer months...

In [4]:
bbox = (-79.91523742675783, 43.508223006137925, -78.82690429687501, 43.806287608590075)
(west, south, east, north)= bbox

print(bbox)

(-79.91523742675783, 43.508223006137925, -78.82690429687501, 43.806287608590075)


In [5]:
start_date = '2025-06-15'
end_date = '2025-08-15'

# Training Data

The data used in this notebook includes

- ECOSTRESS LST: 70m resolution from NASA's ECOSTRESS mission
    this is the main variable we want to downscale...
- Sentinel-2: 70m, and 10m resolution from ESA's Sentinel-2 mission
    first download at 70m to train the model, then at 10m to predict the downscaled LST
- DEM: 30m resolution from NASA's SRTM mission
    elevation data can be useful predictors for LST, so we will include it as an additional variable in our model

For the ECOSTRESS LST data, we will use the `earthaccess` library to search for and download the relevant granules that intersect with our AOI and fall within our specified date range. For the Sentinel-2 data and the DEM data, we can use the `pystac_client`, `odc-stac`, and `Xarray` to search for and download the relevant data.
Unfortunately, ECOSTRESS data is only available via STAC if we are in a cloud environment (e.g. Google Colab) and not on a local machine. So for this notebook, I will demonstrate the process of searching for and downloading the ECOSTRESS LST data using `earthaccess`, and then we can use the downloaded data to train our model and perform the downscaling.

## LST Data

To get the ECOSTRESS LST data, we will use the `earthaccess` library to search for and download the relevant granules that intersect with our AOI and fall within our specified date range. This requires setting up an Earthdata account and configuring the `earthaccess` library with your credentials. Once you have that set up, you can use the following code to search for and download the ECOSTRESS LST data for our AOI and date range.

In [6]:
# Authenticate
import earthaccess

auth = earthaccess.login(persist=True)
print(auth.authenticated)

True


Now let's search for the ECOSTRESS LST data using `earthaccess`. We will specify our AOI, date range, and the collection we want to search in. Then we can download the relevant granules to our local machine.

In [7]:
import warnings
warnings.filterwarnings('ignore')

In [8]:
# Search
import pandas as pd

lst_granules = earthaccess.search_data(
    short_name="ECO_L2T_LSTE",
    version="002",
    bounding_box=(west, south, east, north),
    temporal=(start_date, end_date),
)
print(f"Found {len(lst_granules)} LST granules in the AOI and date range.")

Found 87 LST granules in the AOI and date range.


We could do a bit more filtering here to only download the granules that have good quality data. To do this, we can loop through the LST granules and check for the following conditions before downloading:
- time range between noon and 6 PM (this can be checked on the fly before saving the data, by looking at the timestamp of each granule)
- stream the data (via earthaccess' fsspec) and check the % of valid pixels (not nulls) in the LST band
- if the % of valid pixels is > 75% then save it to disk and download the associated water and QC mask files as well.

This approach runs efficiently and helps us avoid downloading and processing granules that have a lot of missing data, which can save time and storage space. The resulting directory in this case contains only 21 files (data, water masks, QC masks) and is approximately 72 MB in size.

The script for this process is available on GitHub [here](https://github.com/astroAycha/ecostress-lst-downscaling/blob/main/keep_valid_granules.py)

In [9]:
from keep_valid_granules import keep_valid_granules

good_granules = keep_valid_granules(lst_granules)
print(f"\nKept {len(good_granules)} clean granules")


07_17TNJ_20250616T165309_0713_01_LST.tif  valid: 43.7%
 >>> Skipping file: too many Null pixels
07_17TPJ_20250616T165309_0713_01_LST.tif  valid: 3.1%
 >>> Skipping file: too many Null pixels
08_17TNJ_20250616T165401_0713_01_LST.tif  valid: 0.1%
 >>> Skipping file: too many Null pixels
08_17TPJ_20250616T165401_0713_01_LST.tif  valid: 13.7%
 >>> Skipping file: too many Null pixels
07_17TNJ_20250619T160353_0713_01_LST.tif  valid: 0.3%
 >>> Skipping file: too many Null pixels
08_17TPJ_20250619T160445_0713_01_LST.tif  valid: 8.0%
 >>> Skipping file: too many Null pixels
08_17TNJ_20250619T160445_0713_01_LST.tif  valid: 10.8%
 >>> Skipping file: too many Null pixels
Skipping (outside time window): 2025-06-20T15:15:31.157Z
Skipping (outside time window): 2025-06-20T15:15:31.157Z
Skipping (outside time window): 2025-06-20T15:16:23.127Z
Skipping (outside time window): 2025-06-20T15:16:23.127Z
Skipping (outside time window): 2025-06-23T14:27:14.220Z
Skipping (outside time window): 2025-06-23T14:2

QUEUEING TASKS | :   0%|          | 0/3 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/3 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/3 [00:00<?, ?it/s]

09_17TNJ_20250728T185505_0713_01_LST.tif  valid: 99.5%


...

06_17TNJ_20250815T171400_0713_01_LST.tif  valid: 20.9%
 >>> Skipping file: too many Null pixels
07_17TNJ_20250815T171452_0713_01_LST.tif  valid: 58.2%
 >>> Skipping file: too many Null pixels
07_17TPJ_20250815T171452_0713_01_LST.tif  valid: 99.5%


QUEUEING TASKS | :   0%|          | 0/3 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/3 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/3 [00:00<?, ?it/s]


Kept 7 clean granules


Le's take a look at the output of the script

In [10]:
good_granules[0]

{'granule': Collection: {'ShortName': 'ECO_L2T_LSTE', 'Version': '002'}
 Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'BoundingRectangles': [{'WestBoundingCoordinate': -79.76796003845342, 'EastBoundingCoordinate': -78.37385429483595, 'NorthBoundingCoordinate': 44.24655093448751, 'SouthBoundingCoordinate': 43.23597190336682}]}}}
 Temporal coverage: {'RangeDateTime': {'BeginningDateTime': '2025-07-28T18:55:05.134Z', 'EndingDateTime': '2025-07-28T18:55:57.103Z'}}
 Size(MB): 16.72
 Data: ['https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/ECO_L2T_LSTE.002/ECOv002_L2T_LSTE_40034_009_17TPJ_20250728T185505_0713_01/ECOv002_L2T_LSTE_40034_009_17TPJ_20250728T185505_0713_01_water.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/ECO_L2T_LSTE.002/ECOv002_L2T_LSTE_40034_009_17TPJ_20250728T185505_0713_01/ECOv002_L2T_LSTE_40034_009_17TPJ_20250728T185505_0713_01_cloud.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/ECO_L2T_LSTE.002/ECO

### LST Data Processing and Stacking

from IPython.display import HTML

<script src="https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js"></script>
<div class="mermaid">
flowchart LR
    A[Search] --> B[Filter\ndate + cloud]
    B --> C[Filter\ntime + valid pixels]
    C --> D[Download]
    D --> E[QC mask\nwater mask\nclip to AOI]
    E --> F[Stack]
</div>
<script>mermaid.initialize({startOnLoad:true})</script>

Before we use the data, we need to perform some preprocessing steps. This includes QC masking to keep only good quality pixels. We also apply the water mask file to keep only land pixels. Finally we clip the data to our AOI.

The scrip is available on GitHub [here](https://github.com/astroAycha/ecostress-lst-downscaling/blob/main/preprocess_lst.py).
Let's apply it to the LST data we downloaded and see the results.

We need to pause here for a moment to check the data we have. The AOI we chose is not covered by one granule but rather by two tiles. We can infer this from the file names which include `17TNJ` and `17TPJ`. 
`17T` is just the UTM zone. `NJ` and `PJ` are the tile indentifiers in the [Military Grid Reference System](https://en.wikipedia.org/wiki/Military_Grid_Reference_System) (MGRS) tile codes. It's the same tiling system used by Sentinel-2 and HLS. 

In [11]:
# split granules by tile
import xarray


tnj_granules = [g for g in good_granules if "17TNJ" in g["lst_file"]]
tpj_granules = [g for g in good_granules if "17TPJ" in g["lst_file"]]

print(f"17TNJ: {len(tnj_granules)} granules")
print(f"17TPJ: {len(tpj_granules)} granules")

17TNJ: 3 granules
17TPJ: 4 granules


In [12]:
lst_path = good_granules[0]["lst_file"]
qc_path = good_granules[0]["qc_file"]
water_path = good_granules[0]["water_file"]


In [13]:
import rioxarray

lst_ds = rioxarray.open_rasterio(lst_path, masked=True).squeeze("band", drop=True)

In [14]:
lst_ds.rio.crs

CRS.from_wkt('PROJCS["WGS 84 / UTM zone 17N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-81],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32617"]]')

In [15]:
import hvplot.xarray

lst_ds.rio.reproject("EPSG:4326").hvplot.image(x="x", 
                                               y="y", 
                                               rasterize=True, 
                                               cmap="coolwarm", 
                                               title="LST",
                                               frame_width=600,
                                               frame_height=400)

BokehModel(combine_events=True, render_bundle={'docs_json': {'98b03b56-3324-4540-ab12-eaedeac6f226': {'version…

Mask the data using the QC and water masks, and then clip it to our AOI.

- For the QC mask, we will keep pixels with values of 00 (good quality), and 01 (acceeptable quality).

- For the water mask, we will keep pixels with values of 0 (land) and mask out pixels with values of 1 (water).

The full preprocessing script is available on GitHub [here](https://github.com/astroAycha/ecostress-lst-downscaling/blob/main/preprocess_lst.py).

Once we have proccessed the LST data, we can stack the granules and create a mean composite for our AOI. This is the 70 m resolution LST data that we will be one of the entries used to train our model.

In [16]:
from preprocess_lst import qc_mask_lst, clip_and_water_mask_lst
import numpy as np

# Apply the QC and water masks, and clip to the AOI
def build_composite(granules, type="mean", bbox=bbox): 
    """
    Build a composite of the LST data by applying the QC and water masks, 
    and then clipping to the AOI. 
    The composite can be either a mean or median composite, 
    depending on the `type` parameter.

    Parameters:
    ----------
    granules: list of dict
        A list of granules, where each granule is a dictionary containing the 
        paths to the LST, QC, and water files.
    type: str, optional
        The type of composite to build. Can be either "mean" or "median". 
        Default is "mean".

    Returns:
    -------
    xarray.DataArray
        A composite of the LST data, with the same spatial resolution as 
        the input LST data, and a single time dimension.   
    """
    arrays = []
    for item in granules:
        lst_masked = qc_mask_lst(item["lst_file"], item["qc_file"])
        lst_ready = clip_and_water_mask_lst(lst_masked, 
                                            item["water_file"], 
                                            aoi=bbox)
        arrays.append(lst_ready)
    stack = xarray.concat(arrays, dim="time")

    if type == "mean":
        return stack.mean(dim="time", skipna=True)
    elif type == "median":
        return stack.median(dim="time", skipna=True)

We can now build our composite LST. We will build two and then combine them in a way that allows us to deal with the overlap of the two tiles. If we just simply take the mean, the two tiles create edges in the final composite so we will take the mean of the area wheret the two composites to create a smoother final product.

In [17]:
lst_tnj = build_composite(tnj_granules)
lst_tpj = build_composite(tpj_granules)

# reproject TNJ to match TPJ grid
lst_tnj_matched = lst_tnj.rio.reproject_match(lst_tpj)

# where both tiles have data, average them
both_valid = lst_tnj_matched.notnull() & lst_tpj.notnull()
tnj_only   = lst_tnj_matched.notnull() & lst_tpj.isnull()
tpj_only   = lst_tpj.notnull() & lst_tnj_matched.isnull()

lst_mosaic = xarray.where(both_valid, (lst_tnj_matched + lst_tpj) / 2, # average where both valid
                          xarray.where(tnj_only, lst_tnj_matched, lst_tpj) # use TNJ where only it is valid, otherwise TPJ
                        )

lst_mosaic = lst_mosaic.rio.write_crs("EPSG:32617").rio.write_nodata(np.nan)

  QC mask applied >>> 99.5% pixels retained
  QC mask applied >>> 82.8% pixels retained
  QC mask applied >>> 86.8% pixels retained
  QC mask applied >>> 96.1% pixels retained
  QC mask applied >>> 75.6% pixels retained
  QC mask applied >>> 97.8% pixels retained
  QC mask applied >>> 99.5% pixels retained


In [19]:
# save the mosaic
lst_mosaic.rio.to_raster("./products/lst_mosaic_70m.tif", 
                         driver="GTiff",
                         dtype="float32")

In [20]:
lst_mosaic

<xarray.DataArray (y: 499, x: 1081)> Size: 2MB
array([[308.6175 , 308.72998, 309.18   , ...,       nan,       nan,
              nan],
       [308.46167, 308.36334, 308.485  , ...,       nan,       nan,
              nan],
       [308.0833 , 308.08746, 308.205  , ...,       nan,       nan,
              nan],
       ...,
       [309.57083, 309.7475 , 309.2375 , ...,       nan,       nan,
              nan],
       [309.50415, 309.6875 , 309.69418, ...,       nan,       nan,
              nan],
       [309.41168, 309.53915, 309.69418, ...,       nan,       nan,
              nan]], shape=(499, 1081), dtype=float32)
Coordinates:
    band         int64 8B 1
  * x            (x) float64 9kB 6e+05 6.001e+05 ... 6.756e+05 6.756e+05
  * y            (y) float64 4kB 4.853e+06 4.853e+06 ... 4.818e+06 4.818e+06
    spatial_ref  int64 8B 0
Attributes:
    _FillValue:  nan

Let's make a plot of the composite LST to see what it looks like

In [ ]:
import hvplot.xarray

lst_mosaic.rio.reproject("EPSG:4326").hvplot.image(x="x", 
                                                   y="y", 
                                                   rasterize=True, 
                                                   cmap="coolwarm", 
                                                   title="LST", 
                                                   frame_width=600, 
                                                   frame_height=400,
                                                   geo=True)

BokehModel(combine_events=True, render_bundle={'docs_json': {'3f329d07-2c7a-4f8d-92c5-69d6a21731ac': {'version…

Now that we are done with the hard part, let's move on to the next step which is to get the rest of the training data: Sentinel-2 reflectance data and the DEM data in both 70 and 10 m resolution.

## Sentinel-2 data:

We can now use the `pystac_client` and `odc-stac` libraries to search for and download the Sentinel-2 data for our AOI and date range. We will need to specify the collection we want to search in (e.g. "sentinel-2-l2a") and the bands we want to download (e.g. B02, B03, B04, B08). We can also specify the resolution we want to download at (e.g. 10m or 20m).

I went through this process in detail in two previous notebooks [here]() and [here](), so I won't repeat the details again. A lot of what was done for the ECOSTRESS LST data in terms of searching, and aligning the data to the same grid is now done using the `odc-stac` library, which makes it much easier to work with the data in Python.

In [21]:
import pystac_client
import odc.stac

In [22]:
s2_catalog = pystac_client.Client.open("https://earth-search.aws.element84.com/v1")

s2_items = s2_catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=f"{start_date}/{end_date}",
).item_collection()

print(f"Found {len(s2_items)} Sentinel-2 items in the AOI and date range.")

Found 46 Sentinel-2 items in the AOI and date range.


In [23]:
s2_70_ds = odc.stac.load(
    s2_items,
    bbox=bbox,
    bands=["red", "green", "blue", "nir"],
    resolution=70, # <- use the same resolution as the LST data
    chunks={},
    groupby="solar_day",
    crs="EPSG:32617",
    resampling="bilinear",
    query={"eo:cloud_cover": {"lt": 10}}
)

In [24]:
s2_70_ds

<xarray.Dataset> Size: 136MB
Dimensions:      (y: 498, x: 1264, time: 27)
Coordinates:
  * y            (y) float64 4kB 4.853e+06 4.853e+06 ... 4.818e+06 4.818e+06
  * x            (x) float64 10kB 5.873e+05 5.873e+05 ... 6.756e+05 6.757e+05
    spatial_ref  int32 4B 32617
  * time         (time) datetime64[ns] 216B 2025-06-16T16:11:33.794000 ... 20...
Data variables:
    red          (time, y, x) uint16 34MB dask.array<chunksize=(1, 498, 1264), meta=np.ndarray>
    green        (time, y, x) uint16 34MB dask.array<chunksize=(1, 498, 1264), meta=np.ndarray>
    blue         (time, y, x) uint16 34MB dask.array<chunksize=(1, 498, 1264), meta=np.ndarray>
    nir          (time, y, x) uint16 34MB dask.array<chunksize=(1, 498, 1264), meta=np.ndarray>

In [25]:
from rasterio.enums import Resampling

# force S2 onto exactly the same grid as LST
s2_70_ds_matched = s2_70_ds.rio.reproject_match(lst_mosaic, 
                                                resampling=Resampling.average
                                                )

# verify
print("LST shape:", lst_mosaic.shape)
print("S2 shape: ", s2_70_ds_matched["blue"].mean(dim="time").shape)
print("LST bounds:", lst_mosaic.rio.bounds())
print("S2 bounds: ", s2_70_ds_matched.rio.bounds())

LST shape: (499, 1081)
S2 shape:  (499, 1081)
LST bounds: (600000.0, 4817770.0, 675670.0, 4852700.0)
S2 bounds:  (600000.0, 4817770.0, 675670.0, 4852700.0)


In [26]:
import numpy as np

# squeeze time dim and take mean composite 
blue_70 = s2_70_ds_matched["blue"].mean(dim="time").values.astype(float)
green_70 = s2_70_ds_matched["green"].mean(dim="time").values.astype(float)
red_70 = s2_70_ds_matched["red"].mean(dim="time").values.astype(float)
nir_70 = s2_70_ds_matched["nir"].mean(dim="time").values.astype(float)

# DEM data

We do the same for the elevation data. Elevation might not have a big impact for our AOI since Toronto is relatively flat, but it is better to include it as a predictor in our model just in case it does have some influence on the LST values and to be consistent with the approach used in the NASA ARSET training module.

In [27]:
dem_catalog = pystac_client.Client.open(
    "https://earth-search.aws.element84.com/v1"
)

dem_items = dem_catalog.search(
    collections=["cop-dem-glo-30"],
    bbox=bbox
).item_collection()

print(f"Found {len(dem_items)} DEM items")

Found 2 DEM items


In [28]:
import odc.stac

dem_70_ds = odc.stac.load(
    dem_items,
    bbox=bbox,
    bands=["data"],
    resolution=70, # <- use the same resolution as the LST data
    groupby="solar_day",
    chunks={},
    crs="EPSG:32617",
    resampling="bilinear"
)

In [29]:
dem_70_ds

<xarray.Dataset> Size: 3MB
Dimensions:      (y: 498, x: 1264, time: 1)
Coordinates:
  * y            (y) float64 4kB 4.853e+06 4.853e+06 ... 4.818e+06 4.818e+06
  * x            (x) float64 10kB 5.873e+05 5.873e+05 ... 6.756e+05 6.757e+05
    spatial_ref  int32 4B 32617
  * time         (time) datetime64[ns] 8B 2021-04-22
Data variables:
    data         (time, y, x) float32 3MB dask.array<chunksize=(1, 498, 1264), meta=np.ndarray>

In [30]:
# force DEM onto exactly the same grid as LST
dem_70_matched = dem_70_ds.rio.reproject_match(lst_mosaic, 
                                                resampling=Resampling.average
                                                )
# squeeze dem
dem_70 = dem_70_matched["data"].squeeze().values.astype(float)

# verify
print("LST shape:", lst_mosaic.shape)
print("DEM shape: ", dem_70.shape)
print("LST bounds:", lst_mosaic.rio.bounds())
print("DEM bounds: ", dem_70_matched.rio.bounds())

LST shape: (499, 1081)
DEM shape:  (499, 1081)
LST bounds: (600000.0, 4817770.0, 675670.0, 4852700.0)
DEM bounds:  (600000.0, 4817770.0, 675670.0, 4852700.0)


In [31]:
# derive land_mask from the water-masked mosaic
# pixels that survived clip_and_water_mask_lst are land; water pixels are NaN
land_mask = np.isfinite(lst_mosaic.values)
print(f"Land pixels in mosaic: {land_mask.sum()} / {land_mask.size}")

Land pixels in mosaic: 234136 / 539419


In [32]:
print("lst:       ", lst_mosaic.shape)
print("land_mask: ", land_mask.shape)
print("blue:      ", blue_70.shape)
print("green:     ", green_70.shape)
print("red:       ", red_70.shape)
print("nir:       ", nir_70.shape)
print("elev:      ", dem_70.shape)

lst:        (499, 1081)
land_mask:  (499, 1081)
blue:       (499, 1081)
green:      (499, 1081)
red:        (499, 1081)
nir:        (499, 1081)
elev:       (499, 1081)


In [33]:
lst = lst_mosaic.values.astype(float)  # extract numpy array from xarray

# combined mask — land + valid LST + valid S2 + valid DEM
mask = (
    land_mask            &   # water mask
    np.isfinite(lst)     &   # valid LST
    (lst > 0)            &   # no zero LST
    np.isfinite(blue_70)      &   # valid S2
    np.isfinite(green_70)      &
    np.isfinite(red_70)      &
    np.isfinite(nir_70)      &
    np.isfinite(dem_70)        # valid DEM
)

print(f"Valid training pixels after all masks: {mask.sum()}")

Valid training pixels after all masks: 234136


In [34]:
# flatten
def flat(arr):
    return arr[mask].ravel()

X_full = np.column_stack([flat(blue_70), 
                          flat(green_70), 
                          flat(red_70), 
                          flat(nir_70),
                          flat(dem_70)
                          ])
y_full = flat(lst)

print(f"X shape: {X_full.shape}")
print(f"y range: {y_full.min():.1f} — {y_full.max():.1f} K")

X shape: (234136, 5)
y range: 296.0 — 323.5 K


In [35]:
# # don't subsample — use all valid pixels
# X_train, y_train = X_full, y_full
# print(f"Training on all {X_train.shape[0]} valid pixels")

In [36]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_full, 
                                                    y_full, 
                                                    test_size=0.2, 
                                                    random_state=11)
print(f"Training samples: {X_train.shape[0]}, Testing samples: {X_test.shape[0]}")

Training samples: 187308, Testing samples: 46828


In [ ]:
# # S2 nodata is often 0 — treat 0 as invalid
# valid_s2 = (
#     np.isfinite(blue_70) & (blue_70 > 0) &
#     np.isfinite(green_70) & (green_70 > 0) &
#     np.isfinite(red_70) & (red_70 > 0) &
#     np.isfinite(nir_70) & (nir_70 > 0)
# )
# print(f"Valid S2 pixels with >0 check: {valid_s2.sum()}")

In [37]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=50, 
                           random_state=79, 
                           n_jobs=-1)
rf.fit(X_train, y_train)

,n_estimators,50
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [39]:
from sklearn.metrics import mean_absolute_error as mae

y_pred = rf.predict(X_test)
error = mae(y_test, y_pred)
print(f"Mean Absolute Error: {error:.2f} K")

Mean Absolute Error: 1.37 K


# Prediction and evaluation

For the prediction step, we will use a machine learning model to learn the relationship between the low resolution LST data and the high resolution reflectance data. We can use a simple regression model for this, such as a Random Forest or a Gradient Boosting Regressor. We will train the model using the 10m resolution data, and then use it to predict the LST values at 70m resolution using the reflectance data as input.

## to 10m

In [40]:
# load S2 at 10m
s2_10_ds = odc.stac.load(
    s2_items,
    bbox=bbox,
    bands=["blue", "green", "red", "nir"],
    resolution=10,
    chunks={},
    groupby="solar_day",
    crs="EPSG:32617",
    resampling="bilinear",
    query={"eo:cloud_cover": {"lt": 10}}
)


In [41]:
blue_10 = s2_10_ds["blue"].mean(dim="time").values.astype(float)
green_10 = s2_10_ds["green"].mean(dim="time").values.astype(float)
red_10 = s2_10_ds["red"].mean(dim="time").values.astype(float)
nir_10 = s2_10_ds["nir"].mean(dim="time").values.astype(float)

In [42]:
# reproject DEM to 10m
dem_10_matched = dem_70_ds["data"].rio.reproject_match(s2_10_ds,
                                                    resampling=Resampling.bilinear
                                                    )

dem_10 = dem_10_matched.squeeze().values.astype(float)

In [43]:

# valid mask at 10m
mask_10 = (
    np.isfinite(blue_10) & (blue_10 > 0) &
    np.isfinite(green_10) & (green_10 > 0) &
    np.isfinite(red_10) & (red_10 > 0) &
    np.isfinite(nir_10) & (nir_10 > 0) &
    np.isfinite(dem_10)
)

In [44]:
X_pred_10 = np.column_stack([
    blue_10[mask_10], green_10[mask_10], red_10[mask_10], nir_10[mask_10],
    dem_10[mask_10]
])
print(f"Predicting over {X_pred_10.shape[0]} pixels at 10m...")
y_pred_10 = rf.predict(X_pred_10)

Predicting over 30805528 pixels at 10m...


In [45]:
# reconstruct 2D
lst_10m = np.full(mask_10.shape, np.nan)
lst_10m[mask_10] = y_pred_10

print(f"LST 10m shape: {lst_10m.shape}")
print(f"Value range:   {np.nanmin(lst_10m):.1f} — {np.nanmax(lst_10m):.1f} K")

LST 10m shape: (3484, 8842)
Value range:   298.6 — 322.5 K


In [46]:
import xarray as xr

# wrap 10m prediction as DataArray
lst_10m_da = xr.DataArray(
    lst_10m,
    dims=["y", "x"],
    coords={"y": s2_10_ds.y, "x": s2_10_ds.x}
).rio.write_crs("EPSG:32617")

# aggregate prediction back to 70m
lst_pred_agg = lst_10m_da.rio.reproject_match(
    lst_mosaic, resampling=Resampling.average
)

# residual = original 70m LST - aggregated prediction
residual_70m = lst_mosaic - lst_pred_agg

# resample residual to 10m (bilinear = smooth surface)
residual_10m = residual_70m.rio.reproject_match(
    lst_10m_da, resampling=Resampling.bilinear
)

# final bias-corrected 10m LST
lst_final = lst_10m_da + residual_10m

print(f"Final LST 10m range: {float(lst_final.min()):.1f} — {float(lst_final.max()):.1f} K")

Final LST 10m range: 294.0 — 327.7 K


In [47]:
print(f"Residual mean: {float(residual_70m.mean()):.2f} K")
print(f"Residual std:  {float(residual_70m.std()):.2f} K")
print(f"Residual max abs: {float(abs(residual_70m).max()):.2f} K")

Residual mean: 0.14 K
Residual std:  2.05 K
Residual max abs: 14.29 K


In [48]:
# convert to Celsius for easier reading
lst_70m_C = lst_mosaic - 273.15
lst_10m_C = lst_final - 273.15

In [49]:
lst_70m_C.rio.to_raster("./products/lst_70m_C.tif",
                         driver="GTiff",
                            dtype="float32")

In [50]:
lst_10m_C.rio.to_raster("./products/lst_10m_C.tif",
                         driver="GTiff",
                            dtype="float32")

In [51]:
lst_70m_C.hvplot.image(x="x", 
                 y="y", 
                 rasterize=True, 
                 cmap="coolwarm", 
                 title="Downscaled LST at 70m (°C)", 
                 frame_width=600, 
                 frame_height=400,
                 geo=True)

BokehModel(combine_events=True, render_bundle={'docs_json': {'52a1f1d6-7b18-4b82-9df3-cc84ad61a7d6': {'version…

In [52]:
lst_10m_C.hvplot.image(x="x", 
                 y="y", 
                 rasterize=True, 
                 cmap="coolwarm", 
                 title="Downscaled LST at 10m (°C)", 
                 frame_width=600, 
                 frame_height=400,
                 geo=True)

BokehModel(combine_events=True, render_bundle={'docs_json': {'1ca54fd9-4aaf-4374-b614-297aa98ea047': {'version…

## Useful Links:

- [Introduction to Thermal Remote Sensing and Applications in Urban Heat Island Mapping](https://www.earthdata.nasa.gov/learn/trainings/introduction-thermal-remote-sensing-applications-urban-heat-island-mapping)
- [Introduction to `earthaccess`](https://github.com/nasa/LPDAAC-Data-Resources/blob/main/python/tutorials/earthaccess_introduction.ipynb)
- [ECOSTRESS Data Resouces on GitHub](https://github.com/nasa/ECOSTRESS-Data-Resources)
- [Working with ECOSTRESS Tiled Data](https://github.com/nasa/ECOSTRESS-Data-Resources/blob/20dfefbfb793d924aa6b222ddc4c9464e66ff4a6/python/tutorials/Working_with_ECOSTRESS_Tiled_data.ipynb)